<a href="https://colab.research.google.com/github/DanylchenkoKateryna/NLP-Lab-works/blob/main/notebooks/lab2_cleaning_normalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — Cleaning & Normalization Pipeline

**Dataset:** 20 Newsgroups — alt.atheism / sci.electronics / soc.religion.christian  
**Task type:** A — Text Classification  
**Pipeline version:** v2  

### Notebook structure
0. Setup (install deps, mount/clone data)  
1. Load raw data + before-stats  
2. Preprocessing pipeline (`src/preprocess.py`)  
3. Apply pipeline → generate `processed_v2`  
4. Before / After statistics  
5. 15 real before/after examples  
6. Edge cases (30 cases from `tests/edge_cases.jsonl`)  
7. Regression tests (idempotence + no-empty-explosion)  
8. Generate `docs/audit_summary_lab2.md`  

## 0. Setup

In [6]:
# ── 0.1  Install dependencies ─────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'nltk', 'pandas'])

0

In [7]:
# ── 0.2 Detect environment and set REPO_ROOT ─────────────────────────────

import os
import sys
from pathlib import Path

if not os.path.exists('NLP-Lab-works'):
    !git clone https://github.com/DanylchenkoKateryna/NLP-Lab-works.git

%cd NLP-Lab-works

REPO_ROOT = Path.cwd()

sys.path.insert(0, str(REPO_ROOT / 'src'))

print('Repo root:', REPO_ROOT)
print('CWD:', Path.cwd())

Cloning into 'NLP-Lab-works'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 47 (delta 6), reused 43 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 11.26 MiB | 15.44 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/NLP-Lab-works
Repo root: /content/NLP-Lab-works
CWD: /content/NLP-Lab-works


In [8]:
# ── 0.3  Imports ──────────────────────────────────────────────────────────
import json, re, html, unicodedata
from pathlib import Path
from datetime import date

import pandas as pd
import nltk

for resource in ('punkt', 'punkt_tab'):
    try:
        nltk.data.find(f'tokenizers/{resource}')
    except LookupError:
        nltk.download(resource, quiet=True)

from preprocess import clean_text, normalize_text, mask_pii, sentence_split, preprocess, count_replacements

pd.set_option('display.max_colwidth', 120)
print('All imports OK')

All imports OK


## 1. Load Raw Data + Before-Stats

In [9]:
# ── 1.1  Load raw.csv ─────────────────────────────────────────────────────
raw_path = Path('data/raw.csv')
df_raw = pd.read_csv(raw_path)
print(f'Loaded {len(df_raw):,} rows from {raw_path}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head(2)

Loaded 6,383 rows from data/raw.csv
Columns: ['id', 'text', 'subject', 'from', 'category', 'label_id']


,id,text,subject,from,category,label_id
0,0,In <16BB4C522.I3150101@dbstu1.rz.tu-bs.de> I3150101@dbstu1.rz.tu-bs.de (Benedikt Rosenau) writes:\n\n>In article <19...,Re: Islam And Scientific Predictions (was Re: Genocide is Caused by Atheism),darice@yoyo.cc.monash.edu.au (Fred Rice),alt.atheism,0
1,1,In article <May.11.02.37.28.1993.28163@athos.rutgers.edu> dan@ingres.com (a Rose arose) writes:\n\n>4--were God to c...,Re: earthquake prediction,hudson@athena.cs.uga.edu (Paul Hudson Jr),soc.religion.christian,2


In [10]:
# ── 1.2  Before-stats ─────────────────────────────────────────────────────
df_raw['char_len_raw']  = df_raw['text'].fillna('').apply(len)
df_raw['word_count_raw'] = df_raw['text'].fillna('').apply(lambda x: len(x.split()))

print('=== BEFORE stats ===')
print(f"Total docs       : {len(df_raw):,}")
print(f"Empty texts      : {(df_raw['char_len_raw'] == 0).sum()}")
print(f"Very short (<5w) : {(df_raw['word_count_raw'] < 5).sum()}")
print(f"Exact duplicates : {df_raw['text'].duplicated().sum()}")
print()
print('Char length distribution:')
print(df_raw['char_len_raw'].describe().round(1))
print()
print('Word count distribution:')
print(df_raw['word_count_raw'].describe().round(1))

=== BEFORE stats ===
Total docs       : 6,383
Empty texts      : 0
Very short (<5w) : 10
Exact duplicates : 17

Char length distribution:
count     6383.0
mean      1844.2
std       2794.1
min         26.0
25%        713.5
50%       1183.0
75%       2044.0
max      71503.0
Name: char_len_raw, dtype: float64

Word count distribution:
count     6383.0
mean       299.7
std        466.1
min          2.0
25%        108.0
50%        185.0
75%        336.5
max      11679.0
Name: word_count_raw, dtype: float64


## 2. Preprocessing Pipeline

Source: `src/preprocess.py`

```
clean_text()      → remove artifacts, decode HTML, normalize whitespace
normalize_text()  → unicode: apostrophes, quotes, dashes, homoglyphs
mask_pii()        → <URL> <EMAIL> <PHONE>
sentence_split()  → NLTK Punkt English, number-aware
preprocess()      → full pipeline, returns dict
```

In [11]:
# ── 2.1  Quick smoke-test on a single document ────────────────────────────
sample_raw = df_raw['text'].iloc[0]
result = preprocess(sample_raw)

print('=== BEFORE (first 500 chars) ===')
print(repr(sample_raw[:500]))
print()
print('=== AFTER clean ===')
print(repr(result['clean'][:500]))
print()
print(f"Sentences: {result['sentence_count']}")
print(f"Words: {result['word_count']}  |  Chars: {result['char_length']}")

=== BEFORE (first 500 chars) ===
'In <16BB4C522.I3150101@dbstu1.rz.tu-bs.de> I3150101@dbstu1.rz.tu-bs.de (Benedikt Rosenau) writes:\n\n>In article <1993Apr17.122329.21438@monu6.cc.monash.edu.au>\n>darice@yoyo.cc.monash.edu.au (Fred Rice) writes:\n> \n>>>>"AND IT IS HE (GOD ALMIGHTY) WHO CREATED THE NIGHT AND THE\n>>>>DAY, AND THE SUN AND THE EARTH:  ALL (THE CELETIAL BODIES)\n>>>>SWIM ALONG, EACH IN ITS ROUNDED COURSE."  (Holy Quran 21:33)\n>>\n>>>Hmm. This agrees with the Ptolemic system of the earth at the centre,\n>>>with the planets o'

=== AFTER clean ===
'Oops, sorry, my words, not the words of the Qur\'an.\n\nNote that "(the celestial bodies)" in the above verse is an\ninterpolation (which is why it is in brackets) -- it is the translator\'s\n(incorrect, IMHO) interpretation.\n\nHere is Maurice Bucaille\'s translation (he studied Arabic for his\nresearch into the Qur\'an and science) of this verse:\n\n"(God is) the One Who created the night, the day, the sun and the moon.\nEach 

## 3. Apply Pipeline → Generate processed_v2

In [12]:
# ── 3.1  Run pipeline on all documents ───────────────────────────────────
print('Running preprocessing pipeline on all documents...')

results = df_raw['text'].fillna('').apply(preprocess)
results_df = pd.DataFrame(results.tolist())

# Count replacements per document
rep_counts = df_raw['text'].fillna('').apply(count_replacements)
rep_df = pd.DataFrame(rep_counts.tolist())

print(f'Done. Shape: {results_df.shape}')
results_df.head(3)

Running preprocessing pipeline on all documents...
Done. Shape: (6383, 5)


,clean,sentences,sentence_count,char_length,word_count
0,"Oops, sorry, my words, not the words of the Qur'an.\n\nNote that ""(the celestial bodies)"" in the above verse is an\n...","[Oops, sorry, my words, not the words of the Qur'an., Note that ""(the celestial bodies)"" in the above verse is an\ni...",6,787,133
1,"Though there is a command in the law not to heed to one who prophecies\nfalsely, it is still possible for the one wh...","[Though there is a command in the law not to heed to one who prophecies\nfalsely, it is still possible for the one w...",8,730,138
2,"I haven't followed whatever discussion there may have been on these\npeople, but I feel that C. S. Lewis is an excel...","[I haven't followed whatever discussion there may have been on these\npeople, but I feel that C. S. Lewis is an exce...",8,887,149


In [13]:
# ── 3.2  Build processed_v2 DataFrame ────────────────────────────────────
df_v2 = pd.DataFrame({
    'id'            : df_raw['id'],
    'text_v2'       : results_df['clean'],
    'sentence_count': results_df['sentence_count'],
    'char_length'   : results_df['char_length'],
    'word_count'    : results_df['word_count'],
    'label_id'      : df_raw['label_id'],
    'category'      : df_raw['category'],
    'subject'       : df_raw['subject'],
    'n_urls'        : rep_df['urls'],
    'n_emails'      : rep_df['emails'],
    'n_phones'      : rep_df['phones'],
    'n_quote_lines' : rep_df['quote_lines'],
})

# ── 3.3  Save processed_v2 ───────────────────────────────────────────────
out_dir = Path('data/processed_v2')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'processed_v2.csv'
df_v2.to_csv(out_path, index=False)
print(f'Saved {len(df_v2):,} rows to {out_path}')

# ── 3.4  Save sample ─────────────────────────────────────────────────────
sample_dir = Path('data/sample')
sample_dir.mkdir(parents=True, exist_ok=True)
df_v2.head(100).to_csv(sample_dir / 'sample_processed_v2.csv', index=False)
df_raw.head(100).to_csv(sample_dir / 'sample_raw.csv', index=False)
print('Sample files saved (100 rows each)')

Saved 6,383 rows to data/processed_v2/processed_v2.csv
Sample files saved (100 rows each)


## 4. Before / After Statistics

In [14]:
# ── 4.1  Core comparison metrics ─────────────────────────────────────────
before_empty   = (df_raw['char_len_raw'] == 0).sum()
after_empty    = (df_v2['char_length'] == 0).sum()
before_short   = (df_raw['word_count_raw'] < 5).sum()
after_short    = (df_v2['word_count'] < 5).sum()
before_dups    = df_raw['text'].duplicated().sum()
after_dups     = df_v2['text_v2'].duplicated().sum()

total_urls     = rep_df['urls'].sum()
total_emails   = rep_df['emails'].sum()
total_phones   = rep_df['phones'].sum()
total_quotes   = rep_df['quote_lines'].sum()

print('=== COMPARISON: raw vs processed_v2 ===')
print(f"{'Metric':<35} {'Before':>10} {'After':>10}")
print('-' * 57)
print(f"{'Empty texts':<35} {before_empty:>10} {after_empty:>10}")
print(f"{'Very short texts (<5 words)':<35} {before_short:>10} {after_short:>10}")
print(f"{'Exact duplicates':<35} {before_dups:>10} {after_dups:>10}")
print(f"{'Mean char length':<35} {df_raw['char_len_raw'].mean():>10.0f} {df_v2['char_length'].mean():>10.0f}")
print(f"{'Median char length':<35} {df_raw['char_len_raw'].median():>10.0f} {df_v2['char_length'].median():>10.0f}")
print(f"{'Mean word count':<35} {df_raw['word_count_raw'].mean():>10.1f} {df_v2['word_count'].mean():>10.1f}")
print(f"{'Median word count':<35} {df_raw['word_count_raw'].median():>10.0f} {df_v2['word_count'].median():>10.0f}")
print()
print('=== PII REPLACEMENTS MADE ===')
print(f'  URLs masked    : {total_urls:,}')
print(f'  Emails masked  : {total_emails:,}')
print(f'  Phones masked  : {total_phones:,}')
print(f'  Quote lines removed (counted from raw): {total_quotes:,}')

=== COMPARISON: raw vs processed_v2 ===
Metric                                  Before      After
---------------------------------------------------------
Empty texts                                  0          7
Very short texts (<5 words)                 10         28
Exact duplicates                            17       1044
Mean char length                          1844       1263
Median char length                        1183        700
Mean word count                          299.7      216.5
Median word count                          185        117

=== PII REPLACEMENTS MADE ===
  URLs masked    : 2
  Emails masked  : 13,879
  Phones masked  : 1,215
  Quote lines removed (counted from raw): 48,397


In [15]:
# ── 4.2  Distribution of char length before/after ─────────────────────────
bins = [0, 100, 300, 500, 1000, 2000, 5000, 10000, 999999]
labels = ['0-100', '101-300', '301-500', '501-1k', '1k-2k', '2k-5k', '5k-10k', '10k+']

before_dist = pd.cut(df_raw['char_len_raw'], bins=bins, labels=labels).value_counts().sort_index()
after_dist  = pd.cut(df_v2['char_length'],   bins=bins, labels=labels).value_counts().sort_index()

dist_df = pd.DataFrame({'before': before_dist, 'after': after_dist})
dist_df['change'] = dist_df['after'] - dist_df['before']
print('Character length distribution (bucket counts):')
print(dist_df.to_string())

Character length distribution (bucket counts):
         before  after  change
0-100        25    139     114
101-300     338   1048     710
301-500     490   1133     643
501-1k     1751   1827      76
1k-2k      2131   1300    -831
2k-5k      1308    737    -571
5k-10k      248    127    -121
10k+         92     65     -27


In [16]:
# ── 4.3  Sentence count distribution (after only — new metric in v2) ──────
print('Sentence count distribution (processed_v2):')
print(df_v2['sentence_count'].describe().round(1))
print()
print('Docs by sentence count bracket:')
sent_bins = [0, 1, 5, 10, 20, 50, 9999]
sent_labels = ['1', '2-5', '6-10', '11-20', '21-50', '50+']
print(pd.cut(df_v2['sentence_count'], bins=sent_bins, labels=sent_labels).value_counts().sort_index())

Sentence count distribution (processed_v2):
count    6383.0
mean       13.2
std        25.2
min         0.0
25%         4.0
50%         8.0
75%        14.0
max       725.0
Name: sentence_count, dtype: float64

Docs by sentence count bracket:
sentence_count
1         167
2-5      1981
6-10     1987
11-20    1340
21-50     702
50+       199
Name: count, dtype: int64


## 5. 15 Real Before / After Examples

In [17]:
# ── 5.1  Pick 15 diverse examples (some short, some long, across classes) ─
import random
random.seed(42)

# Select: 5 with lots of quote lines, 5 with emails, 5 random
idx_quotes  = rep_df['quote_lines'].nlargest(5).index.tolist()
idx_emails  = rep_df['emails'].nlargest(5).index.tolist()
idx_random  = random.sample(range(len(df_raw)), 5)
selected    = list(dict.fromkeys(idx_quotes + idx_emails + idx_random))[:15]

print(f'Selected {len(selected)} examples for before/after display\n')

for rank, idx in enumerate(selected, 1):
    raw_text    = df_raw.loc[idx, 'text']
    clean_v2    = df_v2.loc[idx, 'text_v2']
    category    = df_raw.loc[idx, 'category']
    raw_words   = df_raw.loc[idx, 'word_count_raw']
    clean_words = df_v2.loc[idx, 'word_count']

    print(f'─── Example {rank:02d} (idx={idx}, class={category}) ───')
    print(f'  BEFORE ({raw_words}w): {repr(raw_text[:300])}')
    print(f'  AFTER  ({clean_words}w): {repr(clean_v2[:300])}')
    print()

Selected 15 examples for before/after display

─── Example 01 (idx=12, class=alt.atheism) ───
  BEFORE (2709w): 'In article <930420.105805.0x8.rusnews.w165w@mantis.co.uk>, mathew <mathew@mantis.co.uk> writes:\n> jbrown@batman.bmd.trw.com writes:\n>>In article <930419.115707.6f2.rusnews.w165w@mantis.co.uk>, mathew\n>><mathew@mantis.co.uk> writes:\n>>> Which "liberal news media" are we talking about?\n>> \n>> Western '
  AFTER  (1345w): 'True. At first, the news media seemed entranced by all the new gizmos\nthe military was using, not to mention the taped video transmissions from\nthe missiles as they zeroed in on their targets. But later, and especially\nafter the bunker full of civilians was hit, they changed their tone. It\nseemed to'

─── Example 02 (idx=2881, class=alt.atheism) ───
  BEFORE (2709w): 'In article <930420.105805.0x8.rusnews.w165w@mantis.co.uk>, mathew <mathew@mantis.co.uk> writes:\n> jbrown@batman.bmd.trw.com writes:\n>>In article <930419.115707.6f2.rusnews.w165w@manti

## 6. Edge Cases

In [18]:
# ── 6.1  Load edge_cases.jsonl ────────────────────────────────────────────
ec_path = Path('tests/edge_cases.jsonl')
edge_cases = []
with open(ec_path, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            edge_cases.append(json.loads(line))

print(f'Loaded {len(edge_cases)} edge cases from {ec_path}')
print('Categories:', {ec['category'] for ec in edge_cases})

Loaded 30 edge cases from tests/edge_cases.jsonl
Categories: {'normalize_text', 'clean_text', 'robustness', 'full_pipeline', 'sentence_split', 'mask_pii'}


In [19]:
# ── 6.2  Run ALL edge cases through the pipeline ──────────────────────────
ec_results = []
for ec in edge_cases:
    result = preprocess(ec['raw_text'])
    ec_results.append({
        'id'               : ec['id'],
        'category'         : ec['category'],
        'expected_behavior': ec['expected_behavior'],
        'raw_text'         : ec['raw_text'],
        'processed'        : result['clean'],
        'sentences'        : result['sentences'],
        'n_sentences'      : result['sentence_count'],
    })

ec_df = pd.DataFrame(ec_results)
print(f'Processed {len(ec_df)} edge cases')
ec_df[['id','category','raw_text','processed']].head(5)

Processed 30 edge cases


,id,category,raw_text,processed
0,EC-01,mask_pii,Contact us at support@example.com for help.,Contact us at <EMAIL> for help.
1,EC-02,mask_pii,Visit https://www.gnu.org/philosophy/philosophy.html for more info.,Visit <URL> for more info.
2,EC-03,mask_pii,Call me at 555-867-5309 if you have questions.,Call me at <PHONE> if you have questions.
3,EC-04,clean_text,From: jsmith@mit.edu (John Smith)\nSubject: Re: question about circuits\n\nHere is the actual message body.,Here is the actual message body.
4,EC-05,clean_text,>In article <1993Apr17.122329@monash.edu>\n>darice@monash.edu (Fred Rice) writes:\n>\n>Some quoted text here.\n\nMy ...,My reply to the above.


In [20]:
# ── 6.3  Show 10 most interesting edge cases ──────────────────────────────
interesting_ids = ['EC-01','EC-04','EC-05','EC-06','EC-07','EC-11',
                   'EC-13','EC-15','EC-24','EC-30']

for row in ec_results:
    if row['id'] in interesting_ids:
        print(f"{'='*60}")
        print(f"ID: {row['id']}  |  Category: {row['category']}")
        print(f"Expected: {row['expected_behavior']}")
        print(f"BEFORE: {repr(row['raw_text'])}")
        print(f"AFTER:  {repr(row['processed'])}")
        if row['category'] == 'sentence_split':
            print(f"Sentences: {row['sentences']}")
        print()

ID: EC-01  |  Category: mask_pii
Expected: Email address replaced with <EMAIL> token
BEFORE: 'Contact us at support@example.com for help.'
AFTER:  'Contact us at <EMAIL> for help.'

ID: EC-04  |  Category: clean_text
Expected: Header lines (From:, Subject:) removed; only body remains
BEFORE: 'From: jsmith@mit.edu (John Smith)\nSubject: Re: question about circuits\n\nHere is the actual message body.'
AFTER:  'Here is the actual message body.'

ID: EC-05  |  Category: clean_text
Expected: All lines starting with > are removed; only the reply remains
BEFORE: '>In article <1993Apr17.122329@monash.edu>\n>darice@monash.edu (Fred Rice) writes:\n>\n>Some quoted text here.\n\nMy reply to the above.'
AFTER:  'My reply to the above.'

ID: EC-06  |  Category: clean_text
Expected: Signature block after '-- ' is removed
BEFORE: 'This is the main message.\n\n--\nJohn Smith\njsmith@example.com\n555-123-4567'
AFTER:  'This is the main message.'

ID: EC-07  |  Category: sentence_split
Expected: Sentence

## 7. Regression Tests

In [21]:
# ── 7.1  Idempotence: preprocess(preprocess(x)["clean"]) == preprocess(x) ─
print('=== Idempotence Test ===')
idempotence_failures = []

# Test on all edge cases + first 200 raw documents
test_texts = [ec['raw_text'] for ec in edge_cases]
test_texts += df_raw['text'].fillna('').head(200).tolist()

for i, text in enumerate(test_texts):
    first_pass  = preprocess(text)['clean']
    second_pass = preprocess(first_pass)['clean']
    if first_pass != second_pass:
        idempotence_failures.append({
            'index': i,
            'first':  repr(first_pass[:100]),
            'second': repr(second_pass[:100]),
        })

if idempotence_failures:
    print(f'FAIL: {len(idempotence_failures)} idempotence failures:')
    for f in idempotence_failures[:5]:
        print(f)
else:
    print(f'PASS: All {len(test_texts)} texts are idempotent ✓')

=== Idempotence Test ===
PASS: All 230 texts are idempotent ✓


In [22]:
# ── 7.2  No empty explosions: non-empty inputs should not produce empty output
# (unless the text was ONLY headers/quotes/signatures)
print('=== No-Empty-Explosion Test ===')
explosion_failures = []

# Non-trivial texts: at least 20 chars in raw and NOT pure metadata
non_trivial = df_raw[df_raw['char_len_raw'] >= 20]['text'].fillna('').tolist()

empty_after = 0
for text in non_trivial:
    result = preprocess(text)['clean']
    if result == '':
        empty_after += 1

pct = 100 * empty_after / len(non_trivial)
print(f'Non-trivial texts tested : {len(non_trivial):,}')
print(f'Became empty after proc  : {empty_after} ({pct:.2f}%)')

if pct < 1.0:
    print('PASS: Empty-explosion rate < 1% ✓')
else:
    print('WARNING: More than 1% of texts became empty — investigate!')

# Also check: absolutely empty input always returns empty output (no crash)
for empty_in in ['', '   ', '\n\t\n', None]:
    try:
        out = preprocess(empty_in if empty_in is not None else '')['clean']
        assert out == '', f'Expected empty string, got {repr(out)}'
    except Exception as e:
        print(f'FAIL on input {repr(empty_in)}: {e}')
print('PASS: Empty/whitespace/None inputs handled safely ✓')

=== No-Empty-Explosion Test ===
Non-trivial texts tested : 6,383
Became empty after proc  : 7 (0.11%)
PASS: Empty-explosion rate < 1% ✓
PASS: Empty/whitespace/None inputs handled safely ✓


## 8. Generate Audit Summary

In [23]:
# ── 8.1  Collect all stats ────────────────────────────────────────────────
raw_mean_chars  = df_raw['char_len_raw'].mean()
v2_mean_chars   = df_v2['char_length'].mean()
raw_mean_words  = df_raw['word_count_raw'].mean()
v2_mean_words   = df_v2['word_count'].mean()
pct_char_drop   = 100 * (1 - v2_mean_chars / raw_mean_chars)
pct_word_drop   = 100 * (1 - v2_mean_words / raw_mean_words)

audit = {
    'date'                    : str(date.today()),
    'pipeline_version'        : 'v2',
    'total_documents'         : len(df_raw),
    'before': {
        'empty_texts'         : int(before_empty),
        'very_short_lt5w'     : int(before_short),
        'exact_duplicates'    : int(before_dups),
        'mean_char_length'    : round(raw_mean_chars, 1),
        'median_char_length'  : int(df_raw['char_len_raw'].median()),
        'mean_word_count'     : round(raw_mean_words, 1),
        'median_word_count'   : int(df_raw['word_count_raw'].median()),
    },
    'after': {
        'empty_texts'         : int(after_empty),
        'very_short_lt5w'     : int(after_short),
        'exact_duplicates'    : int(after_dups),
        'mean_char_length'    : round(v2_mean_chars, 1),
        'median_char_length'  : int(df_v2['char_length'].median()),
        'mean_word_count'     : round(v2_mean_words, 1),
        'median_word_count'   : int(df_v2['word_count'].median()),
        'mean_sentence_count' : round(df_v2['sentence_count'].mean(), 1),
    },
    'replacements': {
        'urls_masked'         : int(total_urls),
        'emails_masked'       : int(total_emails),
        'phones_masked'       : int(total_phones),
        'quote_lines_removed' : int(total_quotes),
    },
    'regression_tests': {
        'idempotence'         : 'PASS' if not idempotence_failures else f'FAIL ({len(idempotence_failures)})',
        'no_empty_explosion'  : 'PASS' if pct < 1.0 else 'WARN',
        'empty_input_safe'    : 'PASS',
    },
    'edge_cases_tested'       : len(edge_cases),
}

print(json.dumps(audit, indent=2))

{
  "date": "2026-03-22",
  "pipeline_version": "v2",
  "total_documents": 6383,
  "before": {
    "empty_texts": 0,
    "very_short_lt5w": 10,
    "exact_duplicates": 17,
    "mean_char_length": 1844.2,
    "median_char_length": 1183,
    "mean_word_count": 299.7,
    "median_word_count": 185
  },
  "after": {
    "empty_texts": 7,
    "very_short_lt5w": 28,
    "exact_duplicates": 1044,
    "mean_char_length": 1263.5,
    "median_char_length": 700,
    "mean_word_count": 216.5,
    "median_word_count": 117,
    "mean_sentence_count": 13.2
  },
  "replacements": {
    "urls_masked": 2,
    "emails_masked": 13879,
    "phones_masked": 1215,
    "quote_lines_removed": 48397
  },
  "regression_tests": {
    "idempotence": "PASS",
    "no_empty_explosion": "PASS",
    "empty_input_safe": "PASS"
  },
  "edge_cases_tested": 30
}


In [24]:
# ── 8.2  Write audit_summary_lab2.md ─────────────────────────────────────
docs_dir = Path('docs')
docs_dir.mkdir(exist_ok=True)

md_lines = [
    '# Audit Summary — Lab 2 Cleaning & Normalization',
    '',
    f"**Generated:** {audit['date']}  ",
    f"**Pipeline version:** {audit['pipeline_version']}  ",
    f"**Total documents:** {audit['total_documents']:,}  ",
    '',
    '## Before vs. After',
    '',
    '| Metric | Before (raw) | After (v2) | Change |',
    '|---|---|---|---|',
    f"| Empty texts | {audit['before']['empty_texts']} | {audit['after']['empty_texts']} | — |",
    f"| Very short (<5 words) | {audit['before']['very_short_lt5w']} | {audit['after']['very_short_lt5w']} | — |",
    f"| Exact duplicates | {audit['before']['exact_duplicates']} | {audit['after']['exact_duplicates']} | — |",
    f"| Mean char length | {audit['before']['mean_char_length']:,} | {audit['after']['mean_char_length']:,} | {pct_char_drop:.0f}% drop |",
    f"| Median char length | {audit['before']['median_char_length']:,} | {audit['after']['median_char_length']:,} | — |",
    f"| Mean word count | {audit['before']['mean_word_count']:,} | {audit['after']['mean_word_count']:,} | {pct_word_drop:.0f}% drop |",
    f"| Median word count | {audit['before']['median_word_count']:,} | {audit['after']['median_word_count']:,} | — |",
    f"| Mean sentence count | N/A | {audit['after']['mean_sentence_count']} | new metric |",
    '',
    '## PII Replacements',
    '',
    f"- URLs masked    : **{audit['replacements']['urls_masked']:,}**",
    f"- Emails masked  : **{audit['replacements']['emails_masked']:,}**",
    f"- Phones masked  : **{audit['replacements']['phones_masked']:,}**",
    f"- Quote lines removed: **{audit['replacements']['quote_lines_removed']:,}**",
    '',
    '## Regression Tests',
    '',
    f"- Idempotence          : `{audit['regression_tests']['idempotence']}`",
    f"- No empty explosion   : `{audit['regression_tests']['no_empty_explosion']}`",
    f"- Empty input safety   : `{audit['regression_tests']['empty_input_safe']}`",
    '',
    '## Edge Cases',
    '',
    f"- Total edge cases tested: **{audit['edge_cases_tested']}**",
    '- Source: `tests/edge_cases.jsonl`',
    '',
    '## Files Generated',
    '',
    '- `data/processed_v2/processed_v2.csv` — full preprocessed dataset',
    '- `data/sample/sample_raw.csv` — 100-row raw sample',
    '- `data/sample/sample_processed_v2.csv` — 100-row processed sample',
]

audit_path = docs_dir / 'audit_summary_lab2.md'
audit_path.write_text('\n'.join(md_lines), encoding='utf-8')
print(f'Audit summary written to {audit_path}')

# Also save JSON version
json_path = docs_dir / 'audit_summary_lab2.json'
json_path.write_text(json.dumps(audit, indent=2), encoding='utf-8')
print(f'JSON audit written to {json_path}')

Audit summary written to docs/audit_summary_lab2.md
JSON audit written to docs/audit_summary_lab2.json


In [25]:
# ── 8.3  Final summary printout ───────────────────────────────────────────
print('=' * 60)
print('  Lab 2 — COMPLETE')
print('=' * 60)
print(f'  Documents processed  : {len(df_v2):,}')
print(f'  Char length drop     : {pct_char_drop:.0f}%')
print(f'  Word count drop      : {pct_word_drop:.0f}%')
print(f'  URLs masked          : {total_urls:,}')
print(f'  Emails masked        : {total_emails:,}')
print(f'  Phones masked        : {total_phones:,}')
print(f'  Quote lines removed  : {total_quotes:,}')
print(f'  Edge cases tested    : {len(edge_cases)}')
print(f'  Idempotence          : {audit["regression_tests"]["idempotence"]}')
print(f'  No-empty-explosion   : {audit["regression_tests"]["no_empty_explosion"]}')
print('=' * 60)
print('Output: data/processed_v2/processed_v2.csv')
print('Audit:  docs/audit_summary_lab2.md')

  Lab 2 — COMPLETE
  Documents processed  : 6,383
  Char length drop     : 31%
  Word count drop      : 28%
  URLs masked          : 2
  Emails masked        : 13,879
  Phones masked        : 1,215
  Quote lines removed  : 48,397
  Edge cases tested    : 30
  Idempotence          : PASS
  No-empty-explosion   : PASS
Output: data/processed_v2/processed_v2.csv
Audit:  docs/audit_summary_lab2.md
